# Actividad: Agente Reactivo Simple

En esta actividad implementaremos y evaluaremos un **agente reactivo simple**.

Un agente reactivo simple selecciona una acción únicamente a partir del estado actual:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

donde:

- $\mathcal{S}$ es el conjunto de estados observables.
- $\mathcal{A}$ es el conjunto de acciones.
- El agente no conserva memoria de estados anteriores.
- El agente no realiza planificación.

El objetivo de la actividad es:

1. Implementar un agente reactivo mediante una política tabular.
2. Implementar un agente aleatorio como referencia.
3. Ejecutar episodios de interacción agente-entorno.
4. Comparar experimentalmente el desempeño de ambos agentes.

In [1701]:
# Importar bibliotecas necesarias
import random

## Entorno: GridWorld

Utilizaremos un entorno discreto representado mediante una cuadrícula de tamaño `size × size`.

### Componentes del entorno

- **Estado:** posición actual del agente `(x, y)`.
- **Estado inicial:** definido por `start`.
- **Estado meta:** definido por `goal`.
- **Acciones posibles:** `UP`, `DOWN`, `LEFT`, `RIGHT`.

El espacio de estados puede representarse como:

$\mathcal{S} = \{(x,y) \mid 0 \leq x,y < size\}$

y el conjunto de acciones como:

$\mathcal{A} = \{UP, DOWN, LEFT, RIGHT\}$

El entorno es determinista: una acción válida produce siempre el movimiento correspondiente.

La función de transición puede representarse como:

$\mathcal{T} : \mathcal{S} \times \mathcal{A} \rightarrow \mathcal{S}$

### Métodos principales

- `reset()` reinicia el entorno al estado inicial.
- `perceive()` devuelve las acciones válidas desde el estado actual.
- `step(action)` ejecuta una acción y actualiza el estado.
- `is_done()` verifica si el agente alcanzó la meta.

In [1702]:
class GridWorld:
    def __init__(self, size=3, start=(0, 0), goal=(2, 2)):
        self.size = size
        self.start = start
        self.goal = goal
        self.agent_pos = start

    def reset(self):
        self.agent_pos = self.start
        return self.agent_pos

    def perceive(self):
        x,y = self.agent_pos
        actions = []

        if x > 0:
          actions.append("LEFT")
        if x < self.size - 1:
          actions.append("RIGHT")
        if y > 0:
          actions.append("UP")
        if y < self.size - 1:
          actions.append("DOWN")
        return actions

    def step(self,action):
        x,y = self.agent_pos

        if action == "UP" and y > 0:
          y = y - 1
        elif action == "DOWN" and y < self.size - 1:
          y = y + 1
        elif action == "LEFT" and x > 0:
          x = x - 1
        elif action == "RIGHT" and x < self.size - 1:
          x = x + 1

        self.agent_pos = (x,y)

    def is_done(self):
        return self.agent_pos == self.goal

## Ejecución de un episodio

Para evaluar un agente necesitamos ejecutar su interacción con el entorno durante varios pasos.

En cada instante:

1. El agente observa el estado actual.
2. Selecciona una acción.
3. El entorno ejecuta la acción.
4. Se actualiza el estado.
5. Se verifica si se alcanzó la meta.

Un episodio termina cuando:

- el agente alcanza la meta, o
- se alcanza un número máximo de pasos.

El límite de pasos evita ejecuciones indefinidas cuando una política no logra alcanzar la meta.

La función `run_episode()` devuelve:

- `success`: indica si el agente alcanzó la meta.
- `steps`: número de acciones ejecutadas.
- `trajectory`: secuencia de estados visitados.

In [1703]:
def run_episode(env, agent, max_steps = 20):
  env.reset()
  trajectory = [env.agent_pos]
  steps = 0

  for step in range(max_steps):
    if env.is_done():
      return True, step, trajectory

    action = agent.act()

    if action is None:
      return False, step, trajectory

    env.step(action)
    trajectory.append(env.agent_pos)

  return env.is_done(), max_steps, trajectory

## Agente Reactivo Simple

Representaremos la política del agente mediante un diccionario.

Un agente reactivo simple implementa una política:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

de modo que a cada estado $s$ se le asigna directamente una acción $a$:

$\pi(s) = a$

El agente:

- observa únicamente el estado actual;
- consulta la acción asociada con ese estado;
- no conserva memoria;
- no realiza búsqueda;
- no analiza estados futuros.

La política contiene, por tanto, conocimiento previamente especificado sobre cómo debe actuar el agente en cada estado.

In [1704]:
# -----------------------------
# Agente Reactivo Simple
# -----------------------------

class SimpleReactiveAgent:
    def __init__(self, env, policy):
        self.env = env
        self.policy = policy

    def act(self):
        return self.policy.get(self.env.agent_pos)

## Política tabular

Definiremos ahora una política para un entorno de tamaño `3 × 3`, con:

- estado inicial `(0, 0)`;
- meta `(2, 2)`.

Cada entrada del diccionario representa una regla de la forma:

`estado → acción`

Por ejemplo:

`(0, 0) → RIGHT`

significa que cuando el agente se encuentre en el estado `(0, 0)`, ejecutará la acción `RIGHT`.

In [1705]:
# Política determinista π : S → A
policy = {
    (0, 0): "RIGHT",
    (0, 1): "RIGHT",
    (0, 2): "RIGHT",

    (1, 0): "RIGHT",
    (1, 1): "RIGHT",
    (1, 2): "RIGHT",

    (2, 0): "DOWN",
    (2, 1): "DOWN",
    (2, 2): None
}

## Prueba del agente reactivo

Ejecutaremos un episodio y observaremos:

- si el agente alcanza la meta;
- cuántos pasos utiliza;
- qué trayectoria sigue.

In [1706]:
env_reactive = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

reactive_agent = SimpleReactiveAgent(
    env_reactive,
    policy
)

success, steps, trajectory = run_episode(
    env_reactive,
    reactive_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: True
Pasos: 4
Trayectoria: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]


## Análisis de la trayectoria

Desde `(0,0)` hasta `(2,2)`, la distancia mínima en una cuadrícula sin obstáculos es:

$d^* = |x_g-x_0| + |y_g-y_0|$

En este caso:

$d^* = |2-0| + |2-0| = 4$

Compara este valor con el número de pasos utilizado por el agente.

Observa que el agente puede seguir una trayectoria óptima aun cuando **no realiza búsqueda ni planificación**.

La trayectoria óptima está implícitamente codificada en la política.

## Agente Aleatorio

Utilizaremos ahora un agente aleatorio como línea base de comparación.

Este agente:

1. obtiene las acciones válidas en el estado actual;
2. selecciona una de ellas al azar.

Si existen $k$ acciones válidas, cada acción tiene probabilidad:

$P(a \mid s) = \frac{1}{k}$

El agente aleatorio:

- no mantiene memoria;
- no realiza planificación;
- no utiliza conocimiento sobre la posición de la meta.

In [1707]:
# -----------------------------
# Agente Aleatorio
# -----------------------------
class RandomAgent:
    def __init__(self, env):
        self.env = env

    def act(self):
        acciones = self.env.perceive()
        if not acciones:
            return None
        return random.choice(acciones)

## Prueba del agente aleatorio

Ejecuta varias veces la siguiente celda.

Observa que el agente puede seguir trayectorias diferentes debido a que la selección de acciones es aleatoria.

In [1708]:
env_random = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

random_agent = RandomAgent(env_random)

success, steps, trajectory = run_episode(
    env_random,
    random_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: False
Pasos: 20
Trayectoria: [(0, 0), (0, 1), (1, 1), (0, 1), (1, 1), (1, 2), (0, 2), (1, 2), (0, 2), (1, 2), (1, 1), (1, 2), (1, 1), (0, 1), (1, 1), (1, 0), (0, 0), (0, 1), (0, 0), (1, 0), (2, 0)]


## Evaluación experimental

Una sola ejecución no es suficiente para comparar agentes, especialmente cuando alguno de ellos tiene comportamiento aleatorio.

Ejecutaremos cada agente durante múltiples episodios y calcularemos dos métricas.

### Tasa de éxito

La tasa de éxito representa la proporción de episodios en los que el agente alcanza la meta:

$\text{Tasa de éxito} =
\frac{\text{episodios exitosos}}
{\text{episodios totales}}$

### Número promedio de pasos

Para los episodios exitosos calcularemos:

$\overline{L} =
\frac{1}{N_s}
\sum_{i=1}^{N_s} L_i$

donde:

- $N_s$ es el número de episodios exitosos;
- $L_i$ es el número de pasos utilizado en el episodio $i$.

Un buen agente debería alcanzar la meta con alta frecuencia y utilizar pocos pasos.

In [1709]:
def evaluate_agent(agent_type, episodes=100, max_steps=20):
    successes = 0
    successful_steps = []

    for _ in range(episodes):

        env = GridWorld(
            size=3,
            start=(0, 0),
            goal=(2, 2)
        )

        if agent_type == "reactive":
            agent = SimpleReactiveAgent(env, policy)

        elif agent_type == "random":
            agent = RandomAgent(env)

        else:
            raise ValueError("Tipo de agente no válido")

        success, steps, _ = run_episode(
            env,
            agent,
            max_steps
        )

        if success:
            successes += 1
            successful_steps.append(steps)

    success_rate = successes / episodes

    if successful_steps:
        avg_steps = sum(successful_steps) / len(successful_steps)
    else:
        avg_steps = None

    return success_rate, avg_steps

## Comparación de los agentes

Ejecutaremos ambos agentes bajo las mismas condiciones:

- mismo tamaño del entorno;
- mismo estado inicial;
- misma meta;
- mismo número máximo de pasos.

El agente reactivo es determinista, por lo que esperamos que repita el mismo comportamiento en todos los episodios.

El agente aleatorio puede producir resultados distintos entre ejecuciones.

In [1710]:
reactive_success, reactive_steps = evaluate_agent(
    "reactive",
    episodes=100,
    max_steps=20
)

random_success, random_steps = evaluate_agent(
    "random",
    episodes=100,
    max_steps=20
)

print("Agente reactivo")
print("Tasa de éxito:", reactive_success)
print("Pasos promedio:", reactive_steps)

print("\nAgente aleatorio")
print("Tasa de éxito:", random_success)
print("Pasos promedio:", random_steps)

Agente reactivo
Tasa de éxito: 1.0
Pasos promedio: 4.0

Agente aleatorio
Tasa de éxito: 0.64
Pasos promedio: 10.0625


In [1711]:
print(f"{'Agente':<15} {'Éxito':<12} {'Pasos promedio'}")
print("-" * 45)

react_str = f"{reactive_steps:.2f}" if reactive_steps is not None else "N/A"
rand_str = f"{random_steps:.2f}" if random_steps is not None else "N/A"

print(
    f"{'Reactivo':<15} "
    f"{reactive_success:<12.2f} "
    f"{react_str}"
)

print(
    f"{'Aleatorio':<15} "
    f"{random_success:<12.2f} "
    f"{rand_str}"
)

Agente          Éxito        Pasos promedio
---------------------------------------------
Reactivo        1.00         4.00
Aleatorio       0.64         10.06


## Preguntas de reflexión

1. ¿Por qué el agente reactivo obtiene siempre el mismo resultado?

    R-. Esto ocurre principalmente debido a que se define la trayectoria y meta del agente en el GridWorld de manera previa, por lo que el camino a recorrer por el agente siempre será el mismo.

2. ¿Por qué el agente aleatorio puede producir trayectorias diferentes?

    R-. Esto es debido a la capacidad del agente aleatorio que tiene para valga la redundancia elegir de manera aletoria movimientos desde el set de acciones para llegar a la meta (***return random.choice(acciones)***), por ende por lo general casi nunca sus rutas serán iguales a la anterior.

3. ¿Alguno de los dos agentes realiza búsqueda en el espacio de estados?

    R-. No, ninguno de los 2 utiliza mecanismos de búsqueda para llegar a la meta, ya que el agente reactivo sigue los pasos predeterminados que le damos para llegar, mientras que el aleotario es exactamente eso, no decido solo realiza movimientos al azar para alcanzar la meta.

4. ¿Por qué el agente reactivo puede encontrar una trayectoria óptima sin utilizar un algoritmo de búsqueda?

    R-. Esto es debido a primera instancia como mencione antes que ya tiene un set de isntrucciones definidas para llegar a la meta, y a su vez no tiene ningun tipo de obstaculo para sortearlas.

5. ¿Qué conocimiento sobre el problema está implícitamente codificado en la política tabular?

    R-. La meta y como llegar a ella de la manera más óptima mediante un set de instrucciones (políticas del agente).

6. ¿Qué ocurriría si cambiamos la posición de la meta pero mantenemos la misma política?

    R-. Pues ya no se volvería ideal y dependiendo de donde este la meta probablemente no pueda llgar a ella nunca.

7. ¿La política tabular utilizada puede considerarse general para cualquier entorno `GridWorld`?

    R-. Si con política nos referimos a la politica diseñada para poder movernos para el mundo, pues no. De hecho en primera instancica se utilizo una politica interpretada como si fuese un plano cartesiano físico, lo cual no se volvio optimo ya que se utilizan y navega de manera diferente a un entorno virtual (caso de el eje "y" en un entorno virtual, que es inverso a un plano real), por lo que no es propicio para cualquier entorno "GridWorld".

8. ¿Qué ventaja ofrece el agente reactivo frente al agente aleatorio?

    R-. La principal ventaja es que este trabaja con un enfoque determinista, teniendo una percepción casi completa de un entorno observable, por ende sabe como llegar de manera óptima a la meta. Donde en cambio, un agente aleatorio solo hace movimientos al azar en base a las acciones y politicas para quizas poder llegar a la meta.

9. ¿Qué limitación importante presenta una política tabular cuando aumenta el número de estados?

    R-. La principal limitante sería la cantidad posible de estados y sus acciones correspondientes, las cuales dependiendo del tamaño pueden ser tantas que se el sistema se vería insostenible/incalculable.

## Reto

Genera de forma pseudoaleatoria un estado inicial y una meta distintos dentro del `GridWorld`.

Después, diseña una política reactiva que permita al agente alcanzar siempre la meta desde cualquier estado inicial.

### Condiciones

- El estado inicial y la meta deben ser diferentes.
- No debes construir manualmente una tabla para cada combinación posible.
- La acción debe depender únicamente del estado actual y de la meta.

### Preguntas

1. ¿Qué reglas utilizaste para decidir el movimiento?

    R-. Las reglas utilizadas fue básicamente comparar la posición actual en la que se econtrara el agente en el GridWorld y compararla con la posición de la meta, donde dependiendo de que tan cerca o lejos se econtrará se moviera acorde a las posiciones x/y de la meta para alcanzarla.
2. ¿El agente realiza búsqueda o planificación?

    R-. El agente realmente realiza una "planificación" pero de manera inmediata, como dice su nombre lo hace de manera reactiva, donde solo evalua su posición actual en relación con la meta y así planea su siguiente movimiento.

3. ¿Por qué esta política funciona para cualquier par inicio--meta en este entorno?

    R-. Esta politica si funcionaria practicamente para casi cualquier combinación de punto de partida y meta, alcanzando la meta siempre debido a que compara su posición con el punto de meta para alcanzar el objetivo en todo momento, siempre y cuando sea un entorno observale y estático.

In [1712]:
# Generar inicio y meta pseudoaleatorios

size = 3

start = (
    random.randrange(size),
    random.randrange(size)
)

goal = start

while goal == start:
    goal = (
        random.randrange(size),
        random.randrange(size)
    )

print("Inicio:", start)
print("Meta:", goal)

Inicio: (1, 2)
Meta: (0, 1)


In [1713]:
class GeneralReactiveAgent:
    def __init__(self, env):
        self.env = env

    def act(self):
        x, y = self.env.agent_pos
        gx, gy = self.env.goal

        # Completar las reglas

        if x < gx:
            return 'RIGHT'
        elif x > gx:
            return 'LEFT'
        elif y < gy:
            return 'DOWN'
        elif y > gy:
            return 'UP'

        return None

env_reto = GridWorld(
    size=size,
    start=start,
    goal=goal
)

Agente_reto = GeneralReactiveAgent(env_reto)

success, steps, trajectory = run_episode(
    env_reto,
    Agente_reto,
    max_steps=20
)

print("Inicio:", start)
print("Meta:", goal)
print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)



# Si se utilizo intelegiencia artifical para la realización de este trabajo a modo de guía para resolver problemas de sintaxis e
# interpretar problemas como la cardinalidad en un entorno virtual, he aquí la conversación: https://share.gemini.google/Qi6SJwmS8k4z

Inicio: (1, 2)
Meta: (0, 1)
Éxito: True
Pasos: 2
Trayectoria: [(1, 2), (0, 2), (0, 1)]
